# 02 — Preparação do dataset para treinamento

Este notebook prepara o **HomeObjects-3K** para o treinamento baseline do YOLOv8n.

Ele parte das decisões tomadas após a auditoria:

- manter as 12 classes no detector;
- preservar o conjunto `val` original como **teste interno final**;
- dividir o `train` original em **treino + validação de desenvolvimento**;
- usar semente fixa para tornar a divisão reproduzível;
- evitar vazamento entre treino e teste causado pelas duplicidades encontradas;
- não modificar nenhum arquivo dentro de `data/raw`;
- gerar um dataset final no padrão YOLO em `data/processed/homeobjects_yolo`.

> **Importante:** este notebook copia os arquivos. O dataset bruto permanece intacto.

## 1. Importações e localização do projeto

A raiz é localizada automaticamente da mesma maneira utilizada no notebook de auditoria.

In [14]:
import random
import shutil
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display


def localizar_raiz_projeto() -> Path:
    pasta_atual = Path.cwd().resolve()

    for candidata in [pasta_atual, *pasta_atual.parents]:
        if (candidata / "requirements.txt").exists() and (candidata / "src").exists():
            return candidata

    raise FileNotFoundError("A raiz do projeto não foi encontrada.")


RAIZ_PROJETO = localizar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from src.auditoria_dados import localizar_raiz_dataset

print(f"Raiz do projeto: {RAIZ_PROJETO}")

Raiz do projeto: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional


## 2. Configurações da preparação

Vamos utilizar:

- **80%** do `train` original para treino;
- **20%** do `train` original para validação;
- o `val` original será reservado para teste;
- semente `42`, já definida no projeto.

A divisão treino/validação tenta preservar a presença das classes, o que é importante porque o dataset é multirrótulo e possui desbalanceamento.

In [15]:
caminho_config = RAIZ_PROJETO / "configs" / "projeto.yaml"

with caminho_config.open(encoding="utf-8") as arquivo:
    configuracao = yaml.safe_load(arquivo)

CLASSES = {int(chave): valor for chave, valor in configuracao["classes"].items()}
SEMENTE = int(configuracao["projeto"].get("semente", 42))
PROPORCAO_VALIDACAO = 0.20

PASTA_RAW = RAIZ_PROJETO / configuracao["dataset"]["pasta_raw"]
RAIZ_DATASET = localizar_raiz_dataset(PASTA_RAW)

if RAIZ_DATASET is None:
    raise FileNotFoundError(
        "O HomeObjects-3K não foi encontrado em data/raw. "
        "Execute primeiro o notebook 01_auditoria_homeobjects.ipynb."
    )

PASTA_TABELAS = RAIZ_PROJETO / "reports" / "tables"
ARQUIVO_DUPLICIDADES = PASTA_TABELAS / "duplicidades.csv"

PASTA_SAIDA = RAIZ_PROJETO / "data" / "processed" / "homeobjects_yolo"
PASTA_SPLITS = RAIZ_PROJETO / "data" / "splits"

print(f"Dataset bruto: {RAIZ_DATASET}")
print(f"Dataset processado: {PASTA_SAIDA}")
print(f"Semente: {SEMENTE}")
print(f"Validação de desenvolvimento: {PROPORCAO_VALIDACAO:.0%}")
print(f"Classes: {len(CLASSES)}")

Dataset bruto: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional\data\raw\homeobjects_extraido
Dataset processado: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional\data\processed\homeobjects_yolo
Semente: 42
Validação de desenvolvimento: 20%
Classes: 12


## 3. Funções auxiliares

Aqui são definidas funções para:

- localizar imagens e labels;
- ler as classes presentes em cada imagem;
- interpretar o relatório de duplicidades;
- criar uma divisão treino/validação que procure manter a representação das classes;
- copiar as imagens e anotações sem alterar o dataset original.

In [16]:
EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def listar_imagens(pasta: Path) -> list[Path]:
    return sorted(
        arquivo
        for arquivo in pasta.rglob("*")
        if arquivo.is_file() and arquivo.suffix.lower() in EXTENSOES_IMAGEM
    )


def caminho_label(caminho_imagem: Path, split_original: str) -> Path:
    relativo = caminho_imagem.relative_to(RAIZ_DATASET / "images" / split_original)
    return (RAIZ_DATASET / "labels" / split_original / relativo).with_suffix(".txt")


def ler_ids_classes(caminho_arquivo_label: Path) -> frozenset[int]:
    ids = set()

    for linha in caminho_arquivo_label.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.strip()

        if not linha:
            continue

        ids.add(int(float(linha.split()[0])))

    return frozenset(ids)


def montar_catalogo(split_original: str) -> pd.DataFrame:
    pasta_imagens = RAIZ_DATASET / "images" / split_original
    registros = []

    for imagem in listar_imagens(pasta_imagens):
        relativo = imagem.relative_to(pasta_imagens)
        label = caminho_label(imagem, split_original)

        if not label.exists():
            raise FileNotFoundError(f"Label não encontrada para: {imagem}")

        ids_classes = ler_ids_classes(label)

        registros.append(
            {
                "split_original": split_original,
                "relativo": relativo.as_posix(),
                "imagem_origem": str(imagem),
                "label_origem": str(label),
                "ids_classes": ids_classes,
                "classes": ", ".join(CLASSES[i] for i in sorted(ids_classes)),
            }
        )

    return pd.DataFrame(registros)


def extrair_split_e_relativo(caminho_texto: str):
    # Caminhos vindos do CSV no Windows possuem uma única barra invertida.
    texto = str(caminho_texto).replace("\\", "/")

    for split in ("train", "val", "test"):
        marcador = f"/images/{split}/"

        if marcador in texto:
            relativo = texto.split(marcador, 1)[1]
            return split, relativo

    return None, None


def identificar_exclusoes_duplicidades(df_duplicidades: pd.DataFrame):
    excluir = set()
    revisao_manual = []

    if df_duplicidades.empty:
        return excluir, pd.DataFrame()

    for _, linha in df_duplicidades.iterrows():
        arquivos = str(linha["arquivos"]).split(" | ")
        itens = []

        for arquivo in arquivos:
            split, relativo = extrair_split_e_relativo(arquivo)

            if split is not None:
                itens.append((split, relativo))

        if len(itens) < 2:
            continue

        tipo_hash = str(linha["tipo_hash"]).strip()
        entre_splits = str(linha["entre_splits"]).strip().lower() == "true"

        # Duplicidade entre desenvolvimento e conjunto oficial de validação:
        # preserva-se o val original, que será nosso teste final.
        if entre_splits:
            for split, relativo in itens:
                if split == "train":
                    excluir.add((split, relativo))

        # Arquivos byte a byte idênticos no mesmo split:
        # mantém-se apenas uma ocorrência.
        elif tipo_hash == "sha256":
            itens_ordenados = sorted(itens)

            for item in itens_ordenados[1:]:
                excluir.add(item)

        # dHash igual dentro do mesmo conjunto não é removido automaticamente.
        else:
            revisao_manual.append(
                {
                    "tipo_hash": tipo_hash,
                    "arquivos": " | ".join(arquivos),
                }
            )

    return excluir, pd.DataFrame(revisao_manual)


def dividir_multirrotulo(
    df: pd.DataFrame,
    proporcao_validacao: float,
    semente: int,
):
    """
    Estratificação iterativa multirrótulo sem dependências externas.

    A função procura preservar, em treino e validação, aproximadamente
    a mesma proporção de imagens de cada classe. Classes raras são
    processadas primeiro para evitar que fiquem concentradas em um split.
    """
    gerador = random.Random(semente)

    total = len(df)
    alvo_val = round(total * proporcao_validacao)
    alvo_train = total - alvo_val

    frequencias = Counter()
    for ids in df["ids_classes"]:
        frequencias.update(ids)

    # Quantidades ainda desejadas por split.
    restantes_total = {
        "train": alvo_train,
        "val": alvo_val,
    }

    restantes_classe = {
        "train": {
            classe_id: frequencias[classe_id] * (1 - proporcao_validacao)
            for classe_id in CLASSES
        },
        "val": {
            classe_id: frequencias[classe_id] * proporcao_validacao
            for classe_id in CLASSES
        },
    }

    nao_atribuidos = set(df.index)
    atribuicoes = {}

    while nao_atribuidos:
        # Conta, entre as amostras ainda não atribuídas,
        # quantas possuem cada classe.
        contagem_restante = Counter()

        for indice in nao_atribuidos:
            contagem_restante.update(df.at[indice, "ids_classes"])

        classes_disponiveis = [
            classe_id
            for classe_id, quantidade in contagem_restante.items()
            if quantidade > 0
        ]

        # Se houver imagens sem rótulos, distribui apenas pela capacidade restante.
        if not classes_disponiveis:
            sobras = list(nao_atribuidos)
            gerador.shuffle(sobras)

            for indice in sobras:
                candidatos = [
                    split for split in ("train", "val")
                    if restantes_total[split] > 0
                ]

                if not candidatos:
                    raise RuntimeError("Não há capacidade restante nos splits.")

                split_escolhido = max(
                    candidatos,
                    key=lambda split: restantes_total[split],
                )

                atribuicoes[indice] = split_escolhido
                restantes_total[split_escolhido] -= 1
                nao_atribuidos.remove(indice)

            break

        # Processa primeiro a classe mais rara entre as amostras restantes.
        menor_frequencia = min(contagem_restante[c] for c in classes_disponiveis)
        classes_mais_raras = [
            c for c in classes_disponiveis
            if contagem_restante[c] == menor_frequencia
        ]
        classe_selecionada = gerador.choice(sorted(classes_mais_raras))

        candidatos_amostras = [
            indice
            for indice in nao_atribuidos
            if classe_selecionada in df.at[indice, "ids_classes"]
        ]
        gerador.shuffle(candidatos_amostras)

        for indice in candidatos_amostras:
            if indice not in nao_atribuidos:
                continue

            splits_disponiveis = [
                split for split in ("train", "val")
                if restantes_total[split] > 0
            ]

            if not splits_disponiveis:
                raise RuntimeError("Não há capacidade restante nos splits.")

            # Primeiro critério: qual split ainda precisa mais da classe rara.
            maior_necessidade_classe = max(
                restantes_classe[split][classe_selecionada]
                for split in splits_disponiveis
            )

            melhores = [
                split for split in splits_disponiveis
                if restantes_classe[split][classe_selecionada]
                == maior_necessidade_classe
            ]

            # Segundo critério: capacidade total restante.
            if len(melhores) > 1:
                maior_capacidade = max(restantes_total[split] for split in melhores)
                melhores = [
                    split for split in melhores
                    if restantes_total[split] == maior_capacidade
                ]

            split_escolhido = gerador.choice(sorted(melhores))

            atribuicoes[indice] = split_escolhido
            restantes_total[split_escolhido] -= 1

            for classe_id in df.at[indice, "ids_classes"]:
                restantes_classe[split_escolhido][classe_id] -= 1

            nao_atribuidos.remove(indice)

    indices_train = [
        indice for indice, split in atribuicoes.items()
        if split == "train"
    ]
    indices_val = [
        indice for indice, split in atribuicoes.items()
        if split == "val"
    ]

    df_train = df.loc[sorted(indices_train)].copy()
    df_val = df.loc[sorted(indices_val)].copy()

    if len(df_train) != alvo_train or len(df_val) != alvo_val:
        raise AssertionError(
            f"Divisão inesperada: train={len(df_train)}, val={len(df_val)}; "
            f"esperado train={alvo_train}, val={alvo_val}."
        )

    return df_train, df_val


def copiar_split(df: pd.DataFrame, split_final: str):
    pasta_imagens_destino = PASTA_SAIDA / "images" / split_final
    pasta_labels_destino = PASTA_SAIDA / "labels" / split_final

    for _, linha in df.iterrows():
        relativo = Path(linha["relativo"])

        imagem_destino = pasta_imagens_destino / relativo
        label_destino = (pasta_labels_destino / relativo).with_suffix(".txt")

        imagem_destino.parent.mkdir(parents=True, exist_ok=True)
        label_destino.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(linha["imagem_origem"], imagem_destino)
        shutil.copy2(linha["label_origem"], label_destino)

## 4. Montagem do catálogo original

O `train` e o `val` originais são lidos sem qualquer modificação.

O `val` original será tratado como **test** a partir deste notebook.

In [17]:
df_train_original = montar_catalogo("train")
df_val_original = montar_catalogo("val")

print(f"Train original: {len(df_train_original)} imagens")
print(f"Val original:   {len(df_val_original)} imagens")
print(f"Total:          {len(df_train_original) + len(df_val_original)} imagens")

display(df_train_original.head())

Train original: 2285 imagens
Val original:   404 imagens
Total:          2689 imagens


,split_original,relativo,imagem_origem,label_origem,ids_classes,classes
0,train,living_room_1.jpg,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,"(1, 3, 4, 10, 11)","sofa, table, lamp, potted plant, photo frame"
1,train,living_room_100.jpg,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,"(1, 2, 3, 4, 5, 7, 10)","sofa, chair, table, lamp, tv, wardrobe, potted..."
2,train,living_room_1001.jpg,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,"(8, 1)","sofa, window"
3,train,living_room_1002.jpg,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,"(1, 10, 3)","sofa, table, potted plant"
4,train,living_room_1003.jpg,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,C:\Users\diego\OneDrive\Documentos_One_Drive\T...,"(1, 10, 3, 7)","sofa, table, wardrobe, potted plant"


## 5. Tratamento conservador das duplicidades

A auditoria encontrou duplicidades por SHA-256 e por dHash.

Neste notebook:

1. **duplicidade entre `train` e `val` original:** preservamos a imagem do `val` (futuro teste) e retiramos a ocorrência do treino;
2. **SHA-256 idêntico no mesmo conjunto:** mantemos apenas uma cópia;
3. **dHash igual dentro do mesmo conjunto:** não removemos automaticamente, pois uma colisão perceptual pode representar imagens legítimas parecidas.

Todas as exclusões são registradas. Nada é apagado de `data/raw`.

In [18]:
if not ARQUIVO_DUPLICIDADES.exists():
    raise FileNotFoundError(
        "O relatório reports/tables/duplicidades.csv não foi encontrado. "
        "Execute e salve primeiro o notebook 01_auditoria_homeobjects.ipynb."
    )

df_duplicidades = pd.read_csv(ARQUIVO_DUPLICIDADES)

excluir, df_revisao_manual = identificar_exclusoes_duplicidades(df_duplicidades)

df_exclusoes = pd.DataFrame(
    sorted(excluir),
    columns=["split_original", "relativo"],
)

print(f"Grupos registrados na auditoria: {len(df_duplicidades)}")
print(f"Arquivos que serão excluídos do dataset processado: {len(df_exclusoes)}")
print(f"Grupos dHash do mesmo split mantidos para revisão: {len(df_revisao_manual)}")

if not df_exclusoes.empty:
    display(df_exclusoes)

if not df_revisao_manual.empty:
    display(df_revisao_manual)

Grupos registrados na auditoria: 20
Arquivos que serão excluídos do dataset processado: 8
Grupos dHash do mesmo split mantidos para revisão: 12


,split_original,relativo
0,train,living_room_1429.jpg
1,train,living_room_1p (203).jpg
2,train,living_room_253.jpg
3,train,living_room_718.jpg
4,train,s (1).jpg
5,train,s (327).jpg
6,train,s (344).jpg
7,val,living_room_1p (201).jpg


,tipo_hash,arquivos
0,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
1,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
2,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
3,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
4,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
5,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
6,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
7,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
8,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...
9,dhash_igual,C:\Users\diego\OneDrive\Documentos_One_Drive\T...


## 6. Aplicação das exclusões

As exclusões afetam somente o dataset que será criado em `data/processed`.

O material original continua preservado em `data/raw`.

In [19]:
def esta_excluido(linha):
    chave = (linha["split_original"], linha["relativo"])
    return chave in excluir


mascara_train = df_train_original.apply(esta_excluido, axis=1)
mascara_test = df_val_original.apply(esta_excluido, axis=1)

df_desenvolvimento = df_train_original.loc[~mascara_train].copy()
df_test = df_val_original.loc[~mascara_test].copy()

print(f"Train original:             {len(df_train_original)}")
print(f"Removidos do train:         {mascara_train.sum()}")
print(f"Base para train + val:      {len(df_desenvolvimento)}")
print()
print(f"Val original:               {len(df_val_original)}")
print(f"Removidos do futuro teste:  {mascara_test.sum()}")
print(f"Teste final:                {len(df_test)}")

Train original:             2285
Removidos do train:         7
Base para train + val:      2278

Val original:               404
Removidos do futuro teste:  1
Teste final:                403


## 7. Criação de treino e validação de desenvolvimento

A base de desenvolvimento é dividida em aproximadamente **80/20**.

Como uma única imagem pode conter várias classes, usamos **estratificação iterativa multirrótulo**.
O algoritmo processa primeiro as classes menos frequentes e procura manter aproximadamente
80% das imagens de cada classe no treino e 20% na validação.

Isso evita situações como uma classe rara ficar quase toda concentrada em apenas um dos conjuntos.

In [20]:
df_train, df_val = dividir_multirrotulo(
    df=df_desenvolvimento,
    proporcao_validacao=PROPORCAO_VALIDACAO,
    semente=SEMENTE,
)

df_train["split_final"] = "train"
df_val["split_final"] = "val"
df_test["split_final"] = "test"

print(f"Treino:     {len(df_train)}")
print(f"Validação:  {len(df_val)}")
print(f"Teste:      {len(df_test)}")
print(f"Total:      {len(df_train) + len(df_val) + len(df_test)}")

Treino:     1822
Validação:  456
Teste:      403
Total:      2681


## 8. Conferência da distribuição das classes

Antes de copiar qualquer arquivo, verificamos quantas imagens de cada classe aparecem em cada conjunto final.

In [21]:
def distribuicao_classes(df, nome_split):
    contador = Counter()

    for ids in df["ids_classes"]:
        contador.update(ids)

    return pd.DataFrame(
        [
            {
                "split": nome_split,
                "id_classe": id_classe,
                "classe": CLASSES[id_classe],
                "imagens_com_classe": contador[id_classe],
                "percentual_do_split": contador[id_classe] / len(df) if len(df) else 0,
            }
            for id_classe in CLASSES
        ]
    )


df_distribuicao = pd.concat(
    [
        distribuicao_classes(df_train, "train"),
        distribuicao_classes(df_val, "val"),
        distribuicao_classes(df_test, "test"),
    ],
    ignore_index=True,
)

tabela_distribuicao = (
    df_distribuicao.pivot(
        index="classe",
        columns="split",
        values="imagens_com_classe",
    )
    .fillna(0)
    .astype(int)
)

tabela_distribuicao["desenvolvimento"] = (
    tabela_distribuicao["train"] + tabela_distribuicao["val"]
)

tabela_distribuicao["pct_val_desenvolvimento"] = (
    tabela_distribuicao["val"] / tabela_distribuicao["desenvolvimento"]
)

display(tabela_distribuicao)

# Nenhuma classe deve desaparecer completamente do treino ou da validação.
classes_sem_train = tabela_distribuicao.index[
    tabela_distribuicao["train"] == 0
].tolist()

classes_sem_val = tabela_distribuicao.index[
    tabela_distribuicao["val"] == 0
].tolist()

if classes_sem_train:
    raise AssertionError(
        f"Classes sem exemplos no treino: {classes_sem_train}"
    )

if classes_sem_val:
    raise AssertionError(
        f"Classes sem exemplos na validação: {classes_sem_val}"
    )

print()
print("Distribuição válida: todas as classes aparecem em treino e validação.")
print("Observe se pct_val_desenvolvimento está próximo de 20% para todas as classes.")

split,test,train,val,desenvolvimento,pct_val_desenvolvimento
classe,,,,,
bed,22,118,29,147,0.197279
chair,154,799,200,999,0.200200
door,58,298,74,372,0.198925
lamp,199,864,216,1080,0.200000
laptop,3,70,17,87,0.195402
photo frame,210,846,211,1057,0.199622
potted plant,317,1256,322,1578,0.204056
sofa,285,1177,294,1471,0.199864
table,299,1323,331,1654,0.200121



Distribuição válida: todas as classes aparecem em treino e validação.
Observe se pct_val_desenvolvimento está próximo de 20% para todas as classes.


## 9. Verificação de sobreposição

Esta etapa garante que nenhuma imagem de origem foi atribuída a mais de um conjunto final.

In [22]:
chaves_train = set(zip(df_train["split_original"], df_train["relativo"]))
chaves_val = set(zip(df_val["split_original"], df_val["relativo"]))
chaves_test = set(zip(df_test["split_original"], df_test["relativo"]))

sobreposicao_train_val = chaves_train & chaves_val
sobreposicao_train_test = chaves_train & chaves_test
sobreposicao_val_test = chaves_val & chaves_test

print(f"Train x Val:  {len(sobreposicao_train_val)}")
print(f"Train x Test: {len(sobreposicao_train_test)}")
print(f"Val x Test:   {len(sobreposicao_val_test)}")

assert not sobreposicao_train_val
assert not sobreposicao_train_test
assert not sobreposicao_val_test

print("OK — não há sobreposição de imagens entre os conjuntos finais.")

Train x Val:  0
Train x Test: 0
Val x Test:   0
OK — não há sobreposição de imagens entre os conjuntos finais.


## 10. Criação do dataset processado

Agora os arquivos são **copiados** para:

```text
data/processed/homeobjects_yolo/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── data.yaml
```

Se a pasta já existir, ela será recriada. Apenas `data/processed/homeobjects_yolo` é apagada; `data/raw` nunca é alterada.

In [23]:
if PASTA_SAIDA.exists():
    shutil.rmtree(PASTA_SAIDA)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_SPLITS.mkdir(parents=True, exist_ok=True)

copiar_split(df_train, "train")
copiar_split(df_val, "val")
copiar_split(df_test, "test")

print("Arquivos copiados com sucesso.")

Arquivos copiados com sucesso.


## 11. Geração do `data.yaml`

O YAML informa ao Ultralytics/YOLO:

- onde está o dataset;
- qual pasta é treino;
- qual pasta é validação;
- qual pasta é teste;
- quais são as 12 classes.

Neste momento o `path` será gravado com o caminho absoluto do Windows. No notebook de treinamento no Colab, esse caminho será ajustado automaticamente para o ambiente da nuvem.

In [24]:
data_yaml = {
    "path": PASTA_SAIDA.resolve().as_posix(),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASSES,
}

caminho_yaml = PASTA_SAIDA / "data.yaml"

with caminho_yaml.open("w", encoding="utf-8") as arquivo:
    yaml.safe_dump(
        data_yaml,
        arquivo,
        allow_unicode=True,
        sort_keys=False,
    )

print(f"YAML criado em: {caminho_yaml}")
print()
print(caminho_yaml.read_text(encoding="utf-8"))

YAML criado em: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional\data\processed\homeobjects_yolo\data.yaml

path: C:/Users/diego/OneDrive/Documentos_One_Drive/TCC/tcc_visao_computacional/data/processed/homeobjects_yolo
train: images/train
val: images/val
test: images/test
names:
  0: bed
  1: sofa
  2: chair
  3: table
  4: lamp
  5: tv
  6: laptop
  7: wardrobe
  8: window
  9: door
  10: potted plant
  11: photo frame



## 12. Salvamento dos registros dos splits

Estes CSVs são importantes para a **reprodutibilidade do TCC**.

Eles permitem demonstrar exatamente quais imagens foram utilizadas em treino, validação e teste.

In [25]:
colunas_exportacao = [
    "split_final",
    "split_original",
    "relativo",
    "classes",
]

df_train[colunas_exportacao].to_csv(
    PASTA_SPLITS / "train.csv",
    index=False,
    encoding="utf-8-sig",
)

df_val[colunas_exportacao].to_csv(
    PASTA_SPLITS / "val.csv",
    index=False,
    encoding="utf-8-sig",
)

df_test[colunas_exportacao].to_csv(
    PASTA_SPLITS / "test.csv",
    index=False,
    encoding="utf-8-sig",
)

df_distribuicao.to_csv(
    PASTA_TABELAS / "distribuicao_classes_splits_finais.csv",
    index=False,
    encoding="utf-8-sig",
)

df_exclusoes.to_csv(
    PASTA_TABELAS / "duplicidades_excluidas_preparacao.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Registros salvos em: {PASTA_SPLITS}")

Registros salvos em: C:\Users\diego\OneDrive\Documentos_One_Drive\TCC\tcc_visao_computacional\data\splits


## 13. Validação final do dataset

A última verificação conta as imagens e labels copiadas e confirma que cada imagem possui uma anotação correspondente.

In [26]:
resumo_final = []

for split in ("train", "val", "test"):
    pasta_imagens = PASTA_SAIDA / "images" / split
    pasta_labels = PASTA_SAIDA / "labels" / split

    imagens = listar_imagens(pasta_imagens)
    labels = list(pasta_labels.rglob("*.txt"))

    resumo_final.append(
        {
            "split": split,
            "imagens": len(imagens),
            "labels": len(labels),
            "diferenca": len(imagens) - len(labels),
        }
    )

df_resumo_final = pd.DataFrame(resumo_final)
display(df_resumo_final)

assert (df_resumo_final["diferenca"] == 0).all()

print("Dataset final validado.")

,split,imagens,labels,diferenca
0,train,1822,1822,0
1,val,456,456,0
2,test,403,403,0


Dataset final validado.


## 14. Resultado da preparação

Ao final desta etapa, teremos:

```text
data/raw/
    HomeObjects-3K original

data/processed/homeobjects_yolo/
    dataset congelado para o experimento

data/splits/
    train.csv
    val.csv
    test.csv

reports/tables/
    duplicidades_excluidas_preparacao.csv
    distribuicao_classes_splits_finais.csv
```

### Decisão metodológica registrada

- As **12 classes** permanecem no modelo de detecção.
- A classificação de risco será desenvolvida em uma etapa posterior e não altera os rótulos do detector.
- O conjunto oficial `val` do HomeObjects-3K foi reservado para o **teste interno final**.
- O `train` original foi dividido de forma reproduzível em treino e validação.
- Possíveis vazamentos identificados pela auditoria foram tratados sem alterar o dataset bruto.

**Próxima etapa:** criar o notebook `03_treinamento_yolov8n_colab.ipynb`.